# 🎨 Stable Diffusion with Tafkir - JDK 25

This notebook demonstrates text-to-image generation using Tafkir's ONNX-based Stable Diffusion implementation running on **pure JDK 25**.

## Why Java for AI?

- **JDK 25**: Project Panama (FFM), Vector API, Records, Pattern Matching
- **No Python overhead**: Native performance with ONNX Runtime
- **Type safety**: Compile-time checking vs Python's runtime errors
- **Production ready**: Enterprise-grade tooling, monitoring, deployment

## Architecture

```
Prompt → CLIP Text Encoder → Text Embeddings
       → UNet (×N steps)   → Denoised Latents
       → VAE Decoder       → RGB Image → PNG
```

All running via ONNX Runtime with Panama FFM bindings.

## 1. Verify Installation

Check that Java 25 and the Tafkir runner are available.

In [1]:
%%bash
echo "🔍 Checking prerequisites..."
echo
echo "Java version:"
java -version 2>&1 | head -1
echo
echo "Tafkir runner:"
RUNNER_JAR="$HOME/Workspace/workkayys/Products/Wayang/wayang-platform/tafkir/ui/tafkir-cli/target/tafkir-runner.jar"
if [ -f "$RUNNER_JAR" ]; then
    SIZE=$(du -h "$RUNNER_JAR" | cut -f1)
    echo "✅ Found ($SIZE)"
else
    echo "❌ Not found. Build first:"
    echo "   cd tafkir/ui/tafkir-cli && mvn clean package -DskipTests"
fi
echo
echo "ONNX Runtime:"
if [ -f "/opt/homebrew/lib/libonnxruntime.dylib" ] || [ -f "/usr/local/lib/libonnxruntime.dylib" ]; then
    echo "✅ Installed"
else
    echo "⚠️  Not found. Install: brew install onnxruntime"
fi

EvalException: Undefined cell magic 'bash'

## 2. Check Model Availability

Verify the Stable Diffusion model is downloaded.

In [ ]:
%%bash
echo "📂 Checking for Stable Diffusion model..."
echo
MODEL_DIR="$HOME/.tafkir/models/blobs"
if [ -d "$MODEL_DIR" ]; then
    echo "✅ Model directory exists"
    echo
    echo "Looking for SD components:"
    find "$MODEL_DIR" -name "model.onnx" -type f 2>/dev/null | head -5
    echo
    echo "Model size:"
    du -sh "$MODEL_DIR" 2>/dev/null
else
    echo "❌ Model not found. Download with:"
    echo "   java -jar tafkir-runner.jar pull CompVis/stable-diffusion-v1-4 --branch onnx"
fi

## 3. Generate Your First Image

Run Stable Diffusion with a simple prompt. This will take 1-3 minutes depending on your hardware.

In [2]:
%%bash
RUNNER_JAR="$HOME/Workspace/workkayys/Products/Wayang/wayang-platform/tafkir/ui/tafkir-cli/target/tafkir-runner.jar"

java -jar "$RUNNER_JAR" run \
    --model CompVis/stable-diffusion-v1-4 \
    --branch onnx \
    --prompt "a cat playing ball in a sunny garden, photorealistic" \
    --seed 42 \
    --steps 10 \
    --guidance-scale 7.5 \
    --output cat-playing.png

echo
echo "📊 Output file info:"
if [ -f "cat-playing.png" ]; then
    file cat-playing.png
    ls -lh cat-playing.png
    echo
    echo "✅ Image saved successfully!"
fi

EvalException: Undefined cell magic 'bash'

## 4. Display the Generated Image

In [ ]:
from IPython.display import Image, display
import os

if os.path.exists("cat-playing.png"):
    print("✅ Generated Image:")
    display(Image(filename="cat-playing.png", width=512))
else:
    print("❌ Image not found. Run the previous cell first.")

## 5. Generate Multiple Variations

Create different images by changing the seed while keeping the prompt constant.

In [ ]:
%%bash
RUNNER_JAR="$HOME/Workspace/workkayys/Products/Wayang/wayang-platform/tafkir/ui/tafkir-cli/target/tafkir-runner.jar"
PROMPT="a cyberpunk cat with neon lights, digital art, synthwave aesthetic"

echo "🎨 Generating 3 variations..."
echo "Prompt: $PROMPT"
echo

for seed in 100 200 300; do
    echo "---"
    echo "Generating with seed=$seed..."
    
    java -jar "$RUNNER_JAR" run \
        --model CompVis/stable-diffusion-v1-4 \
        --branch onnx \
        --prompt "$PROMPT" \
        --seed $seed \
        --steps 10 \
        --output "variation-seed-${seed}.png" 2>&1 | grep -E "(✓|❌|Time)"
    
    echo
done

echo "✅ All variations generated!"
ls -lh variation-seed-*.png 2>/dev/null

## 6. Display All Variations

In [ ]:
from IPython.display import Image, display, HTML
import os

seeds = [100, 200, 300]
print("🎨 Generated Variations:")
print()

html = '<div style="display: flex; gap: 10px; flex-wrap: wrap;">'
for seed in seeds:
    filename = f"variation-seed-{seed}.png"
    if os.path.exists(filename):
        html += f'<div><img src="{filename}" width="256" /><br/><center><b>Seed: {seed}</b></center></div>'
    else:
        html += f'<div style="width:256px;height:256px;background:#eee;display:flex;align-items:center;justify-content:center;">Not generated</div>'
html += '</div>'

display(HTML(html))

## 7. Experiment with Guidance Scale

See how the CFG (Classifier-Free Guidance) parameter affects the output.

In [3]:
%%bash
RUNNER_JAR="$HOME/Workspace/workkayys/Products/Wayang/wayang-platform/tafkir/ui/tafkir-cli/target/tafkir-runner.jar"
PROMPT="a majestic dragon flying over mountains at sunset"

echo "🔬 Testing guidance scales..."
echo "Prompt: $PROMPT"
echo

for cfg in 5.0 7.5 10.0; do
    echo "---"
    echo "CFG=$cfg..."
    
    java -jar "$RUNNER_JAR" run \
        --model CompVis/stable-diffusion-v1-4 \
        --branch onnx \
        --prompt "$PROMPT" \
        --seed 42 \
        --steps 10 \
        --guidance-scale $cfg \
        --output "cfg-${cfg}.png" 2>&1 | grep -E "(✓|❌)"
done

echo
echo "✅ Guidance scale comparison complete!"

EvalException: Undefined cell magic 'bash'

## 8. Display Guidance Comparison

In [ ]:
from IPython.display import Image, display, HTML
import os

cfgs = [5.0, 7.5, 10.0]
print("🔬 Guidance Scale Comparison:")
print()

html = '<div style="display: flex; gap: 10px; flex-wrap: wrap;">'
for cfg in cfgs:
    filename = f"cfg-{cfg}.png"
    if os.path.exists(filename):
        html += f'<div><img src="{filename}" width="256" /><br/><center><b>CFG: {cfg}</b></center></div>'
html += '</div>'

display(HTML(html))

print()
print("💡 Observations:")
print("- Lower CFG (5.0): More creative, less prompt adherence")
print("- Medium CFG (7.5): Balanced - recommended default")
print("- Higher CFG (10.0): Strict prompt following, may be oversaturated")

## 9. Performance Benchmark

Test how generation time scales with step count.

In [ ]:
%%bash
RUNNER_JAR="$HOME/Workspace/workkayys/Products/Wayang/wayang-platform/tafkir/ui/tafkir-cli/target/tafkir-runner.jar"

echo "⏱️  Performance Benchmark: Steps vs Time"
echo
printf "%-10s | %-15s\n" "Steps" "Time (sec)"
printf "%-10s-|-%-15s\n" "----------" "---------------"

for steps in 5 10 15; do
    START=$(date +%s)
    
    java -jar "$RUNNER_JAR" run \
        --model CompVis/stable-diffusion-v1-4 \
        --branch onnx \
        --prompt "a cat" \
        --seed 42 \
        --steps $steps \
        --output "bench-${steps}.png" >/dev/null 2>&1
    
    END=$(date +%s)
    DURATION=$((END - START))
    
    printf "%-10s | %-15s\n" "$steps" "${DURATION}s"
done

## 10. Summary & Next Steps

### What We Learned
✅ Generate images from text prompts using JDK 25  
✅ Control randomness with seed parameter  
✅ Adjust quality vs speed with step count  
✅ Control prompt adherence with guidance scale  
✅ Create multiple variations efficiently  
✅ Benchmark performance characteristics  

### JDK 25 Features Used
- **Project Panama**: FFM bindings for ONNX Runtime
- **Vector API**: SIMD-accelerated tensor operations
- **Records**: Clean data structures
- **Pattern Matching**: Type-safe inference results

### Next Steps
- Try different prompts and styles
- Experiment with higher step counts (30-50) for quality
- Test various guidance scales (5-15)
- Generate batches for dataset creation

### Resources
- [Tafkir Documentation](https://tafkir-ai.github.io)
- [Stable Diffusion Tutorial](https://tafkir-ai.github.io/docs/tutorials/intermediate/stable-diffusion/)
- [GitHub Repository](https://github.com/tafkir-ai/tafkir)

---

**Running Java on AI? Yes, it's possible and performant!** 🚀